<div dir="rtl">
<h1>هر سطر Logits باید هدف خودش را داشته باشد</h1>
<p>درس 49 از 76 · نمایش Cتایی چگونه V امتیاز می‌سازد؟ · <code dir="ltr">43-lm-head</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/43-lm-head.html">📖 بازگشت به همین درس</a></p>
<p>خروجی Language-model head و Loss را با حفظ تطابق Batch/Time بسازید.</p><p>پیش‌نیاز: Linear، Logits، Cross-Entropy و هدف یک‌خانه‌جلو را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>برای B=2 و T=3، هدفِ (1,0) پس از صاف‌کردن در کدام سطر است؟ اگر فقط Target را Transpose کنیم، Shape خطا را آشکار می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
model = MiniGPT(ModelConfig(7,6,8,2,1,0.)).eval()
hidden = torch.randn(2,3,8)
targets = torch.tensor([[1,2,3],[4,5,6]])
print('head weight:',model.language_model_head.weight.shape,'targets:',targets)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع prediction_loss(hidden, Weight, targets) زوج (Logits, Loss) برگرداند. Weight همان وزن (V,C) بدون Bias است؛ Logits را خودتان ضرب کنید و برای Loss، B و T را در هر دو ورودی با ترتیب یکسان صاف کنید. پیش از F.cross_entropy، Softmax نزنید.</p>
</div>

In [ ]:
def prediction_loss(hidden, weight, targets):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = prediction_loss(hidden,model.language_model_head.weight,targets)
    if result is None: return False
    logits,loss = result
    torch.testing.assert_close(logits,model.language_model_head(hidden))
    separate = torch.stack([F.cross_entropy(logits[b,t][None],targets[b,t][None]) for b in range(2) for t in range(3)]).mean()
    torch.testing.assert_close(loss,separate)
    x = torch.randn(1,4,3); w = torch.randn(5,3); y = torch.tensor([[0,1,2,3]])
    z,l = prediction_loss(x,w,y)
    torch.testing.assert_close(z,x@w.T)
    torch.testing.assert_close(l,F.cross_entropy(z.reshape(-1,5),y.reshape(-1)))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Vocabulary را از ۷ به ۱۰ افزایش دهید؛ C ثابت است. اینجا فقط تعداد امتیازها و Parameterهای Head را مقایسه می‌کنیم، نه کیفیت مدل‌های تصادفی تازه را.</p>
</div>

In [ ]:
from torch import nn
for V in (7,10):
    head = nn.Linear(8,V,bias=False)
    print('V, output shape, parameters:',V,head(hidden).shape,sum(p.numel() for p in head.parameters()))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>دادهٔ کنترل‌شدهٔ زیر برای هر سطر، امتیاز کلاس درست را بالا گذاشته است. شش کلاس دیگر هرکدام ۸ واحد امتیاز کمتر دارند؛ بنابراین Loss درست هر سطر حدود ۰٫۰۰۲ است. کد خراب Target را با ترتیب زمان-اول صاف می‌کند. تابع aligned_loss(Logits,targets) را اصلاح کنید.</p>
</div>

In [ ]:
fixture = torch.full((2,3,7),-4.)
fixture.scatter_(-1,targets[...,None],4.)
wrong = F.cross_entropy(fixture.reshape(-1,7),targets.T.reshape(-1))
correct = fixture.new_tensor(math.log1p(6*math.exp(-8.)))
print('wrong ordering/known expected loss:',wrong.item(),correct.item())
assert wrong.item() > 1.

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def aligned_loss(logits, targets):
    # TODO
    return None

In [ ]:
def test_repair():
    result = aligned_loss(fixture,targets)
    if result is None: return False
    torch.testing.assert_close(result,correct)
    assert result.item() < 0.01
    logits = torch.tensor([[[1.,2.],[3.,-1.]]])
    y = torch.tensor([[1,0]])
    torch.testing.assert_close(aligned_loss(logits,y),F.cross_entropy(logits[0],y[0]))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>language_model_head و محاسبهٔ نهایی Loss در MiniGPT همین قرارداد را دارند. hidden واقعی مدل پس از final_norm می‌آید؛ اینجا نمایش کوچکِ آماده دادیم تا خطای تطابق سطرها جدا دیده شود.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا Loss عددی و Shape درست برای تأیید جفت‌شدن هر پیش‌بینی با هدف درست کافی نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/43-lm-head.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/43-lm-head.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>